# Distilling the Li$_3$OCl teacher from MACE-MP-0

The foundation model MACE-MP-0 (small) labels configurations from short molecular dynamics at three temperatures and configurations dragged through the vacancy hop. A MACE with 64 invariant channels, two interaction layers, correlation order three and a 5 Å cutoff is trained on them (`mace_run_train`).

1. labelled configurations from the foundation model
2. training of the teacher
3. held-out errors and speed of the teacher, single and batched

Runtime: GPU. Output: `li3ocl_teacher.model` and the training set `li3ocl_train.xyz`.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from ase import Atoms, units
from ase.io import write, read
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import mace_mp, MACECalculator
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; A_EQ = 3.926
unit = Atoms('ClOLi3', scaled_positions=[(0,0,0), (.5,.5,.5), (.5,.5,0), (.5,0,.5), (0,.5,.5)], cell=[A_EQ]*3, pbc=True)
def build():
    s = unit.repeat((3,3,3)); li = [i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li']; vac = s.positions[li[0]].copy(); del s[li[0]]; return s, vac
base, VAC = build(); fm = mace_mp(model='small', device=DEV, default_dtype='float32')
def label(a):
    a = a.copy(); a.calc = fm; e, f = a.get_potential_energy(), a.get_forces(); a.calc = None
    a.info = {'REF_energy': float(e)}; a.arrays['REF_forces'] = f.copy(); return a
frames = []

In [ ]:
# (1a) thermal configurations from short foundation-model MD
t0 = time.time()
for T in (700, 1000, 1300):
    at = base.copy(); at.calc = fm; MaxwellBoltzmannDistribution(at, temperature_K=T, rng=np.random.default_rng(T))
    dyn = Langevin(at, 2 * units.fs, temperature_K=T, friction=0.02 / units.fs); dyn.run(300)
    for k in range(120):
        dyn.run(15); s = at.copy(); s.calc = None; s.info = {'REF_energy': float(at.get_potential_energy())}; s.arrays['REF_forces'] = at.get_forces().copy(); frames.append(s)
    print(T, 'K done,', len(frames), 'frames,', round(time.time() - t0), 's')

In [ ]:
# (1b) configurations along the vacancy hop: drag each of the 8 nearest Li towards the vacant site, with thermal rattling
cell = base.cell.lengths(); li = np.array([i for i, z in enumerate(base.get_chemical_symbols()) if z == 'Li'])
d = base.positions[li] - VAC; d -= cell * np.round(d / cell); near = li[np.argsort(np.linalg.norm(d, axis=1))[:8]]
rng = np.random.default_rng(0)
for i in near:
    vec = VAC - base.positions[i]; vec -= cell * np.round(vec / cell)
    for lam in np.linspace(0.1, 0.9, 9):
        for rep in range(3):
            a = base.copy(); a.positions[i] += lam * vec; a.positions += rng.normal(0, 0.06, a.positions.shape); frames.append(label(a))
print(len(frames), 'frames in total'); rng.shuffle(frames)
write('li3ocl_train.xyz', frames[:-60]); write('li3ocl_test.xyz', frames[-60:])

In [ ]:
# (2) train the mid-size teacher.  If a flag is rejected by the installed mace version, the error message names it.
!mace_run_train --name=li3ocl_teacher --train_file=li3ocl_train.xyz --valid_fraction=0.08 --test_file=li3ocl_test.xyz \
  --energy_key=REF_energy --forces_key=REF_forces --E0s=average --model=MACE --hidden_irreps='64x0e' --r_max=5.0 --num_interactions=2 --correlation=3 \
  --batch_size=8 --valid_batch_size=8 --max_num_epochs=120 --lr=0.01 --ema --ema_decay=0.99 --amsgrad --forces_weight=100 --energy_weight=1 \
  --device=cuda --default_dtype=float32 --seed=1 --save_cpu 2>&1 | tail -25
!ls -la *.model checkpoints 2>/dev/null | head

In [ ]:
import glob
MODEL = sorted(glob.glob('li3ocl_teacher*.model'), key=len)[0]; print('using', MODEL)
tc = MACECalculator(model_paths=MODEL, device=DEV, default_dtype='float32')
test = read('li3ocl_test.xyz', ':'); eF, eE = [], []
for a in test:
    b = a.copy(); b.calc = tc; eF.append(b.get_forces() - a.arrays['REF_forces']); eE.append((b.get_potential_energy() - a.info['REF_energy']) / len(a))
F_RMSE = float(np.sqrt(np.mean(np.concatenate(eF) ** 2)) * 1e3); E_RMSE = float(np.std(eE) * 1e3)
print(f'teacher vs foundation model: force RMSE {F_RMSE:.1f} meV/A, energy RMSE {E_RMSE:.2f} meV/atom (constant shift removed)')

In [ ]:
# (3) speed: plain ASE MD, and the model alone for batches of replicas
at = base.copy(); at.calc = tc; MaxwellBoltzmannDistribution(at, temperature_K=1000); dyn = Langevin(at, 2 * units.fs, temperature_K=1000, friction=0.02 / units.fs)
dyn.run(50); t0 = time.time(); dyn.run(500); MD_MS = (time.time() - t0) / 500 * 1e3; print(f'ASE MD: {MD_MS:.1f} ms/step = {86400 / MD_MS * 1e3 * 2e-6:.1f} ns/day')
from mace import data
from mace.tools import torch_geometric, utils
model = tc.models[0]; zt = utils.AtomicNumberTable([int(z) for z in model.atomic_numbers]); BATCH = []
for B in (1, 4, 16, 32, 64, 128):
    try:
        ds = []
        for b in range(B):
            a = base.copy(); a.rattle(0.08, seed=b); ds.append(data.AtomicData.from_config(data.config_from_atoms(a), z_table=zt, cutoff=float(model.r_max)))
        d = next(iter(torch_geometric.dataloader.DataLoader(dataset=ds, batch_size=B, shuffle=False, drop_last=False))).to(DEV).to_dict()
        for _ in range(3): model(d, compute_force=True)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(10): model(d, compute_force=True)
        torch.cuda.synchronize(); ms = (time.time() - t0) / 10 * 1e3; BATCH.append(dict(B=B, ms_per_replica=round(ms / B, 2))); print(BATCH[-1])
    except RuntimeError as e:
        print('B =', B, 'failed:', str(e)[:100]); break

In [ ]:
print('=========== SUMMARY ===========')
print(json.dumps(dict(gpu=torch.cuda.get_device_name(0), n_train=len(frames) - 60, teacher_force_rmse_meVA=round(F_RMSE, 1), teacher_energy_rmse_meV_atom=round(E_RMSE, 2),
                      ase_md_ms_per_step=round(MD_MS, 1), batched=BATCH), indent=1))
print('==============================')
!zip -q li3ocl_step3.zip li3ocl_train.xyz li3ocl_test.xyz li3ocl_teacher*.model
from google.colab import files; files.download('li3ocl_step3.zip')